In [1]:
!pip install numpy


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import numpy as np

# data = np.load("rod_centerline.npz")
data = np.load("example.npz")
# xs = data["rod_qs"]
# m1s = data["rod_m1s"]
xs = data["true_qs"]
m1s = data["true_m1s"]

print(xs.shape, m1s.shape) # (10, 123) (10, 30, 3)
time_array = np.linspace(0.0, 1, xs.shape[0])
# append time_array as first column
dof_with_time = np.hstack((time_array.reshape(-1, 1), xs))
print(dof_with_time.shape) # (10, 124)

(11, 123) (11, 30, 3)
(11, 124)


In [3]:
import numpy as np

def map_node_to_dof(node_nums: int | np.ndarray) -> np.ndarray:
        return (3 * np.asarray(node_nums))[..., None] + np.array([0, 1, 2])

def logDataForRendering(dof_with_time, n_nodes, Nsteps, mapNodetoDOF, ms):
    n_rod_nodes = n_nodes

    # For dynamic case
    rod_data = np.zeros((n_rod_nodes * Nsteps, 4))
    m1_data =  0.0 * np.zeros(( (n_rod_nodes-1) * Nsteps, 3))
    for i in range(Nsteps):
        for j in range(n_rod_nodes):
            rod_data[i * n_rod_nodes + j, 0] = dof_with_time[i, 0]
            rod_data[i * n_rod_nodes + j, 1:] = dof_with_time[i, 1 + mapNodetoDOF(j)]
        for k in range(n_rod_nodes-1): # n_edges
            m1_data[i* (n_rod_nodes-1) + k, :] = ms[i, k, :]
    
    # print(m1_data.shape) # (Nsteps * (n_rod_nodes-1), 3
    # print(m1_data)

    np.savetxt('rawDataRod.txt', rod_data, fmt='%.6e')
    np.savetxt('rawDataM1.txt', m1_data, fmt='%.6e')

    return rod_data, m1_data

def export_rod_shell_data(n_nodes, rod_file='rawDataRod.txt', m1_file='rawDataM1.txt', rod_js='rodData.js',
                          rod_radius=0.1, scaleFactor=100):
    """
    Export rod and shell data to .js files for visualization.
    """

    # === Load rod data ===
    df = np.loadtxt(rod_file)
    dm = np.loadtxt(m1_file)
    n_rod_nodes = n_nodes
    
    # find length of first edge
    print(df[1, 1:], df[0, 1:])
    len_edge = np.linalg.norm(df[1, 1:] - df[0, 1:]) * scaleFactor
    scale_m = len_edge / 10.0
    print(f'Scale for m1 vectors: {scale_m}')

    # Write rod data
    with open(rod_js, 'w') as fileID:
        fileID.write(f'nNodes = {n_rod_nodes};\n')
        fileID.write(f'rodRadius = {rod_radius};\n')
        fileID.write('nodesRod = [\n')

        for row in df:
            t, x, y, z = row
            x, y, z = x * scaleFactor, y * scaleFactor, z * scaleFactor
            fileID.write(f'{t}, 1, {x}, {y}, {z},\n')

        fileID.write(']\n;\n')

        fileID.write('m1Rod = [\n')
        for row in dm:
            m1x, m1y, m1z = row
            m1x, m1y, m1z = m1x * scale_m, m1y * scale_m, m1z * scale_m
            fileID.write(f'{m1x}, {m1y}, {m1z},\n')
        fileID.write(']\n;\n')
        



In [ ]:
logDataForRendering(dof_with_time, n_nodes=31, Nsteps=dof_with_time.shape[0], mapNodetoDOF=map_node_to_dof, ms = m1s)
# export_rod_shell_data(n_nodes = 31)
export_rod_shell_data(n_nodes = 31, rod_file='rawDataRod.txt', m1_file='rawDataM1.txt', rod_js='rodData_true.js')

[0.004163 0.015861 0.      ] [0.000833 0.015861 0.      ]
Scale for m1 vectors: 0.0333
